# OpenDSS Verification: psforge-flow ACPF vs OpenDSS Power Flow

This notebook compares AC power flow results between:
- **psforge-flow** (Newton-Raphson, positive-sequence model) — pre-computed results
- **OpenDSS** (via `opendssdirect.py`, three-phase balanced model) — computed live

The comparison validates that psforge-grid's `DSSWriter` produces correct OpenDSS scripts
by verifying that the exported `.dss` files yield consistent power flow solutions.

## Dependencies

```bash
pip install psforge-grid  # includes opendssdirect.py
```

In [1]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

from psforge_grid import System

## 1. Load Pre-computed psforge-flow Results

psforge-flow ACPF results were computed separately and saved as JSON.
This avoids requiring psforge-flow as a dependency for this notebook.

In [2]:
NOTEBOOK_DIR = Path(__file__).parent if "__file__" in dir() else Path.cwd()
# Resolve paths relative to the notebook location
RESULTS_FILE = NOTEBOOK_DIR / "psforge_flow_acpf_results.json"
if not RESULTS_FILE.exists():
    RESULTS_FILE = Path("notebooks/psforge_flow_acpf_results.json")

FIXTURES = NOTEBOOK_DIR.parent / "tests" / "fixtures"
if not FIXTURES.exists():
    FIXTURES = Path("tests/fixtures")

with open(RESULTS_FILE) as f:
    pf_results = json.load(f)

print(f"Available systems: {list(pf_results.keys())}")
for name, data in pf_results.items():
    print(
        f"  {name}: {len(data['buses'])} buses, {len(data['branches'])} branches, "
        f"converged={data['converged']}, iterations={data['iterations']}"
    )

Available systems: ['case5_pjm', 'ieee14']
  case5_pjm: 5 buses, 6 branches, converged=True, iterations=5
  ieee14: 14 buses, 20 branches, converged=True, iterations=3


## 2. Helper Functions

### Accuracy metric rationale

| Quantity | Metric | Rationale |
|----------|--------|-----------|
| **\|V\|** (voltage mag.) | Absolute diff [pu] | Always near 1.0 pu; 0.002 pu is directly interpretable |
| **θ** (voltage angle) | Absolute diff [deg] | Can be zero (swing bus); % is undefined or misleading |
| **P** (active power) | Absolute diff [MW] | Spans wide range including zero-crossings; % diverges near zero |
| **Q** (reactive power) | Absolute diff [Mvar] | Frequently crosses zero; model differences make % misleading |

For P and Q, we also show the difference normalized by system base MVA (pu) to provide
a scale-independent view across different system sizes.

In [3]:
def update_generators_from_results(system: System, pf_data: dict) -> System:
    """Update generator P/Q values with converged psforge-flow results.

    MATPOWER files store scheduled (not converged) generator Q values.
    PSS/E RAW files store converged values. To ensure fair comparison,
    we override generator P/Q with the converged values from psforge-flow.

    The results JSON stores per-generator values (distributed proportionally
    by original P when multiple generators share the same bus).
    """
    if "generators" not in pf_data:
        return system

    pf_gens = pf_data["generators"]
    # Match generators by position (same order as system.generators)
    for i, gen in enumerate(system.generators):
        if i < len(pf_gens):
            gen.p_gen = pf_gens[i]["p_gen_pu"]
            gen.q_gen = pf_gens[i]["q_gen_pu"]

    return system


def run_opendss_powerflow(system: System) -> dict:
    """Export System to .dss, run OpenDSS power flow in subprocess, and extract results.

    Uses subprocess isolation to avoid segfaults from consecutive OpenDSS Compile calls.
    """
    with tempfile.TemporaryDirectory() as tmpdir:
        dss_path = Path(tmpdir) / "system.dss"
        system.to_dss(dss_path)
        safe_path = str(dss_path.resolve()).replace("\\", "/")

        script = f'''
import json, math
import opendssdirect as dss

dss.Basic.ClearAll()
r = dss.run_command('Compile "{safe_path}"')
if r and "error" in r.lower():
    raise RuntimeError(f"OpenDSS compile error: {{r}}")
dss.run_command("Solve Mode=Snapshot")
if not dss.Solution.Converged():
    raise RuntimeError("OpenDSS power flow did not converge")

buses = []
for name in dss.Circuit.AllBusNames():
    dss.Circuit.SetActiveBus(name)
    pm = dss.Bus.puVmagAngle()
    if len(pm) >= 2:
        buses.append({{"bus_name": name, "v_mag_pu": pm[0], "v_angle_deg": pm[1]}})

branches = []
flag = dss.Lines.First()
while flag > 0:
    powers = dss.CktElement.Powers()
    p = (powers[0]+powers[2]+powers[4]) if len(powers)>=6 else 0
    q = (powers[1]+powers[3]+powers[5]) if len(powers)>=6 else 0
    branches.append({{"name": dss.Lines.Name(), "type": "line", "p_flow_kw": p, "q_flow_kvar": q}})
    flag = dss.Lines.Next()

flag = dss.Transformers.First()
while flag > 0:
    powers = dss.CktElement.Powers()
    p = (powers[0]+powers[2]+powers[4]) if len(powers)>=6 else 0
    q = (powers[1]+powers[3]+powers[5]) if len(powers)>=6 else 0
    branches.append({{"name": dss.Transformers.Name(), "type": "transformer", "p_flow_kw": p, "q_flow_kvar": q}})
    flag = dss.Transformers.Next()

print(json.dumps({{"buses": buses, "branches": branches}}))
'''
        result = subprocess.run(
            [sys.executable, "-c", script],
            capture_output=True,
            text=True,
            timeout=60,
        )
        if result.returncode != 0:
            raise RuntimeError(f"OpenDSS subprocess failed: {result.stderr}")
        return json.loads(result.stdout.strip())


def compare_bus_voltages(pf_buses, dss_buses):
    """Compare bus voltages: |V| in absolute pu, angle in absolute degrees."""
    import re

    dss_lookup = {b["bus_name"].lower(): b for b in dss_buses}

    print(
        f"{'Bus':<12} {'|V|_pf':>8} {'|V|_dss':>8} {'dV[pu]':>9}  "
        f"{'ang_pf':>8} {'ang_dss':>8} {'da[deg]':>8}"
    )
    print("-" * 70)

    max_v_diff_pu = 0.0
    max_a_diff_deg = 0.0

    for pf_bus in pf_buses:
        name = pf_bus["name"].lower()
        sanitized = re.sub(r"[\s.\-/\\]+", "_", name)
        sanitized = re.sub(r"[^a-z0-9_]", "", sanitized)
        if sanitized and sanitized[0].isdigit():
            sanitized = f"n{sanitized}"

        dss_bus = dss_lookup.get(name) or dss_lookup.get(sanitized)
        if dss_bus is None:
            print(f"{pf_bus['name']:<12} {'NOT FOUND':>8}")
            continue

        v_pf = pf_bus["v_mag_pu"]
        v_dss = dss_bus["v_mag_pu"]
        v_diff = abs(v_pf - v_dss)

        a_pf = pf_bus["v_angle_deg"]
        a_dss = dss_bus["v_angle_deg"]
        a_diff = abs(a_pf - a_dss)

        max_v_diff_pu = max(max_v_diff_pu, v_diff)
        max_a_diff_deg = max(max_a_diff_deg, a_diff)

        v_flag = " *" if v_diff > 0.01 else ""
        print(
            f"{pf_bus['name']:<12} {v_pf:>8.5f} {v_dss:>8.5f} {v_diff:>8.5f}{v_flag}  "
            f"{a_pf:>8.3f} {a_dss:>8.3f} {a_diff:>8.4f}"
        )

    print("-" * 70)
    print(f"Max |V| difference: {max_v_diff_pu:.5f} pu")
    print(f"Max angle difference: {max_a_diff_deg:.4f} deg")
    return max_v_diff_pu, max_a_diff_deg


def compare_branch_flows(pf_branches, dss_branches, base_mva):
    """Compare branch flows: absolute MW/Mvar difference and normalized by base MVA."""
    print(
        f"{'Branch':<12} {'P_pf':>8} {'P_dss':>8} {'dP[MW]':>8} {'dP[pu]':>8}"
        f"  {'Q_pf':>8} {'Q_dss':>8} {'dQ[Mv]':>8} {'dQ[pu]':>8}"
    )
    print("-" * 95)

    max_p_diff_mw = 0.0
    max_q_diff_mvar = 0.0

    dss_lines = [b for b in dss_branches if b["type"] == "line"]
    dss_xfmrs = [b for b in dss_branches if b["type"] == "transformer"]
    dss_ordered = dss_lines + dss_xfmrs

    n = min(len(pf_branches), len(dss_ordered))
    for i in range(n):
        pf_br = pf_branches[i]
        dss_br = dss_ordered[i]

        p_pf = pf_br["p_flow_pu"] * base_mva
        q_pf = pf_br["q_flow_pu"] * base_mva
        p_dss = dss_br["p_flow_kw"] / 1000.0
        q_dss = dss_br["q_flow_kvar"] / 1000.0

        dp_mw = abs(p_pf - p_dss)
        dq_mvar = abs(q_pf - q_dss)
        dp_pu = dp_mw / base_mva
        dq_pu = dq_mvar / base_mva

        max_p_diff_mw = max(max_p_diff_mw, dp_mw)
        max_q_diff_mvar = max(max_q_diff_mvar, dq_mvar)

        label = pf_br.get("name", f"{pf_br['from_bus']}-{pf_br['to_bus']}")
        p_flag = " *" if dp_pu > 0.005 else ""
        q_flag = " *" if dq_pu > 0.005 else ""
        print(
            f"{label:<12} {p_pf:>8.2f} {p_dss:>8.2f} {dp_mw:>8.3f} {dp_pu:>7.5f}{p_flag}"
            f"  {q_pf:>8.2f} {q_dss:>8.2f} {dq_mvar:>8.3f} {dq_pu:>7.5f}{q_flag}"
        )

    print("-" * 95)
    print(f"Max P difference: {max_p_diff_mw:.3f} MW ({max_p_diff_mw / base_mva:.5f} pu)")
    print(f"Max Q difference: {max_q_diff_mvar:.3f} Mvar ({max_q_diff_mvar / base_mva:.5f} pu)")
    return max_p_diff_mw, max_q_diff_mvar

## 3. Case 1: pglib-opf case5_pjm (5-bus system)

Small system for initial validation.

In [4]:
system_5 = System.from_matpower(FIXTURES / "pglib_opf_case5_pjm.m")
update_generators_from_results(system_5, pf_results["case5_pjm"])
print(f"System: {len(system_5.buses)} buses, {len(system_5.branches)} branches")

dss_result_5 = run_opendss_powerflow(system_5)
print(f"OpenDSS: {len(dss_result_5['buses'])} buses, {len(dss_result_5['branches'])} branches")

System: 5 buses, 6 branches


OpenDSS: 5 buses, 6 branches


In [5]:
print("=== Bus Voltage Comparison: case5_pjm ===")
pf5 = pf_results["case5_pjm"]
v_diff_5, a_diff_5 = compare_bus_voltages(pf5["buses"], dss_result_5["buses"])

=== Bus Voltage Comparison: case5_pjm ===
Bus            |V|_pf  |V|_dss    dV[pu]    ang_pf  ang_dss  da[deg]
----------------------------------------------------------------------
Bus1          1.00000  1.00000  0.00000     1.205    1.205   0.0000
Bus2          0.98938  0.98938  0.00000    -2.425   -2.425   0.0002
Bus3          1.00000  1.00000  0.00000    -2.004   -2.004   0.0001
Bus4          1.00000  1.00000  0.00000     0.000   -0.000   0.0000
Bus5          1.00000  1.00000  0.00000     1.905    1.905   0.0000
----------------------------------------------------------------------
Max |V| difference: 0.00000 pu
Max angle difference: 0.0002 deg


In [6]:
print("=== Branch Flow Comparison: case5_pjm ===")
p_diff_5, q_diff_5 = compare_branch_flows(
    pf5["branches"], dss_result_5["branches"], pf5["base_mva"]
)

=== Branch Flow Comparison: case5_pjm ===
Branch           P_pf    P_dss   dP[MW]   dP[pu]      Q_pf    Q_dss   dQ[Mv]   dQ[pu]
-----------------------------------------------------------------------------------------------
1-2            225.19   225.19    0.008 0.00008     21.98    21.98    0.003 0.00003
1-4             68.58    68.58    0.002 0.00002     -6.46    -6.46    0.001 0.00001
1-5           -188.77  -188.77    0.005 0.00005     18.48    18.48    0.001 0.00001
2-3            -76.24   -76.24    0.006 0.00006    -90.31   -90.31    0.004 0.00004
3-4           -116.40  -116.39    0.007 0.00007     13.36    13.36    0.002 0.00002
4-5           -110.63  -110.63    0.001 0.00001     12.59    12.59    0.001 0.00001
-----------------------------------------------------------------------------------------------
Max P difference: 0.008 MW (0.00008 pu)
Max Q difference: 0.004 Mvar (0.00004 pu)


## 4. Case 2: IEEE 14-bus system

Standard test case with transformers and multiple voltage levels.

In [7]:
system_14 = System.from_raw(FIXTURES / "ieee14.raw")
update_generators_from_results(system_14, pf_results["ieee14"])
print(f"System: {len(system_14.buses)} buses, {len(system_14.branches)} branches")

dss_result_14 = run_opendss_powerflow(system_14)
print(f"OpenDSS: {len(dss_result_14['buses'])} buses, {len(dss_result_14['branches'])} branches")

System: 14 buses, 20 branches


OpenDSS: 14 buses, 20 branches


In [8]:
print("=== Bus Voltage Comparison: IEEE 14-bus ===")
pf14 = pf_results["ieee14"]
v_diff_14, a_diff_14 = compare_bus_voltages(pf14["buses"], dss_result_14["buses"])

=== Bus Voltage Comparison: IEEE 14-bus ===
Bus            |V|_pf  |V|_dss    dV[pu]    ang_pf  ang_dss  da[deg]
----------------------------------------------------------------------
Bus 1         1.06000  1.06000  0.00000     0.000   -0.000   0.0000
Bus 2         1.04500  1.04458  0.00042    -4.983   -4.996   0.0130
Bus 3         1.01000  1.00932  0.00068   -12.725  -12.756   0.0313
Bus 4         1.01767  1.01680  0.00087   -10.313  -10.347   0.0338
Bus 5         1.01951  1.01868  0.00083    -8.774   -8.804   0.0302
Bus 6         1.07000  1.06839  0.00161   -14.221  -14.310   0.0894
Bus 7         1.06152  1.06028  0.00124   -13.360  -13.424   0.0649
Bus 8         1.09000  1.08879  0.00121   -13.360  -13.424   0.0648
Bus 9         1.05593  1.05451  0.00143   -14.938  -15.020   0.0818
Bus 10        1.05099  1.04951  0.00148   -15.097  -15.182   0.0846
Bus 11        1.05691  1.05533  0.00158   -14.791  -14.879   0.0888
Bus 12        1.05519  1.05352  0.00167   -15.076  -15.169   0.0938


In [9]:
print("=== Branch Flow Comparison: IEEE 14-bus ===")
p_diff_14, q_diff_14 = compare_branch_flows(
    pf14["branches"], dss_result_14["branches"], pf14["base_mva"]
)

=== Branch Flow Comparison: IEEE 14-bus ===
Branch           P_pf    P_dss   dP[MW]   dP[pu]      Q_pf    Q_dss   dQ[Mv]   dQ[pu]
-----------------------------------------------------------------------------------------------
1-2            156.88   157.44    0.556 0.00556 *    -20.40   -19.80    0.602 0.00602 *
1-5             75.51    75.79    0.280 0.00280      3.85     4.22    0.361 0.00361
2-3             73.24    73.36    0.119 0.00119      3.56     3.68    0.122 0.00122
2-4             56.13    56.35    0.216 0.00216     -1.55    -1.34    0.209 0.00209
2-5             41.52    41.71    0.196 0.00196      1.17     1.36    0.189 0.00189
3-4            -23.29   -23.19    0.095 0.00095      4.47     4.55    0.077 0.00077
4-5            -61.16   -61.22    0.066 0.00066     15.82    15.75    0.070 0.00070
6-11             7.35     7.32    0.030 0.00030      3.56     3.55    0.014 0.00014
6-12             7.79     7.80    0.017 0.00017      2.50     2.51    0.008 0.00008
6-13          

## 5. Summary

In [10]:
print("=" * 70)
print("Verification Summary: psforge-flow vs OpenDSS")
print("=" * 70)

base5 = pf5["base_mva"]
base14 = pf14["base_mva"]

# --- Tolerances ---
TOL_V_PU = 0.005  # |V|: 0.005 pu
TOL_A_DEG = 0.15  # angle: 0.15 deg
TOL_P_PU = 0.01  # P: 0.01 pu (1 MW on 100 MVA base)
TOL_Q_PU = 0.01  # Q: 0.01 pu (1 Mvar on 100 MVA base)

print()
print("--- Per-system results ---")
print(f"{'System':<14} {'dV[pu]':>10} {'da[deg]':>10} {'dP[pu]':>10} {'dQ[pu]':>10}")
print("-" * 58)
print(
    f"{'case5_pjm':<14} {v_diff_5:>10.5f} {a_diff_5:>10.4f} "
    f"{p_diff_5 / base5:>10.5f} {q_diff_5 / base5:>10.5f}"
)
print(
    f"{'IEEE 14-bus':<14} {v_diff_14:>10.5f} {a_diff_14:>10.4f} "
    f"{p_diff_14 / base14:>10.5f} {q_diff_14 / base14:>10.5f}"
)

print()
print("--- Acceptance criteria ---")
print()
checks = {
    "|V|": (max(v_diff_5, v_diff_14), TOL_V_PU, "pu"),
    "angle": (max(a_diff_5, a_diff_14), TOL_A_DEG, "deg"),
    "P": (max(p_diff_5 / base5, p_diff_14 / base14), TOL_P_PU, "pu"),
    "Q": (max(q_diff_5 / base5, q_diff_14 / base14), TOL_Q_PU, "pu"),
}

all_pass = True
for name, (val, tol, unit) in checks.items():
    status = "PASS" if val <= tol else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  {name:<8} max={val:.5f} {unit:<4} tol={tol:.5f} {unit:<4} -> {status}")

print()
print(f"Overall verification: {'PASS' if all_pass else 'FAIL'}")

Verification Summary: psforge-flow vs OpenDSS

--- Per-system results ---
System             dV[pu]    da[deg]     dP[pu]     dQ[pu]
----------------------------------------------------------
case5_pjm         0.00000     0.0002    0.00008    0.00004
IEEE 14-bus       0.00167     0.0938    0.00556    0.00602

--- Acceptance criteria ---

  |V|      max=0.00167 pu   tol=0.00500 pu   -> PASS
  angle    max=0.09379 deg  tol=0.15000 deg  -> PASS
  P        max=0.00556 pu   tol=0.01000 pu   -> PASS
  Q        max=0.00602 pu   tol=0.01000 pu   -> PASS

Overall verification: PASS


## Notes

### Accuracy metric choice

| Quantity | Metric | Why not % |
|----------|--------|-----------|
| **\|V\|** | Absolute [pu] | V is always near 1.0 pu, so absolute and % are nearly equivalent. Absolute pu is standard in power systems. |
| **theta** | Absolute [deg] | Swing bus angle = 0; denominator is zero. % is undefined. |
| **P, Q** | Absolute [MW, Mvar] and normalized [pu on Sbase] | Flows span from ~0 to hundreds of MW and frequently cross zero. A 0.001 MW error at near-zero flow produces absurdly large %. Normalizing by Sbase gives scale-independent comparison. |

### Acceptance criteria

| Quantity | Tolerance | Physical meaning |
|----------|-----------|------------------|
| \|V\| | 0.005 pu | Half of typical 1% voltage regulation band |
| theta | 0.15 deg | Within typical measurement accuracy |
| P | 0.01 pu | 1% of system capacity (1 MW on 100 MVA base) |
| Q | 0.01 pu | 1% of system capacity (1 Mvar on 100 MVA base) |

### Model differences

- psforge-flow uses a **positive-sequence (single-phase equivalent)** Newton-Raphson method
- OpenDSS uses a **three-phase balanced** current injection / Newton-Raphson method
- Under balanced three-phase conditions, both approaches should yield similar results

### DSSWriter configuration

- Circuit source (Vsource) uses `Mvasc3=1e10` for near-ideal voltage source behavior
- Generators use `model=1` (constant PQ) with converged reactive power values from psforge-flow
- Per-generator P/Q values are distributed proportionally by original P when multiple generators share a bus
- Transformer losses use `%loadloss` (not `%R`) to correctly set both winding resistances

### Root cause analysis of IEEE 14-bus discrepancies

**case5_pjm (no transformers)**: Near-perfect match (dV < 0.00001 pu) confirms that transmission line modeling and generator injection are consistent between the two tools.

**IEEE 14-bus (3 transformers)**: Small but measurable differences (max dV ≈ 0.002 pu) traced to two causes:

1. **`%R` vs `%loadloss` bug (fixed)**: OpenDSS `%R` property only sets the *active* winding's resistance. After array properties like `kvs`/`kvas` shift the active winding, `%R=0` only zeroed one winding while leaving the other at its default (0.2%). Fixed by using `%loadloss=0` which correctly sets both windings. This reduced max dV from ~0.0023 to ~0.0017 pu.

2. **Inherent 3-phase vs positive-sequence transformer model difference (~0.001 pu)**: Verified with minimal 2-bus test systems:
   - Line element: dV = 0.000007 pu (essentially perfect)
   - Transformer element: dV ≈ 0.001 pu (inherent cross-tool difference)

   OpenDSS models transformers as 3-phase coupled inductors with per-phase winding connections, while psforge-flow uses a positive-sequence pi-equivalent circuit. Even under balanced conditions, small numerical differences arise from these fundamentally different internal representations.

3. **Cascading Q redistribution**: Each transformer's small Q difference (~0.2–0.3 Mvar) accumulates (~0.65 Mvar total), causing the Vsource to compensate with ~0.96 Mvar of additional Q injection. This propagates voltage changes throughout the network, amplifying the per-transformer differences.

### Conclusion

The remaining ~0.002 pu voltage difference for the IEEE 14-bus system is an **inherent cross-tool modeling difference**, not a bug. Systems without transformers (case5_pjm) show near-perfect agreement, confirming the correctness of the DSSWriter implementation.